# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook shows how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished} | Version: {metadata.version}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their fields, and their IDs, using the dataset's schema.

In [ ]:
# List all record sets, fields, and their @id
for record_set in metadata.recordSets:
    print(f"RecordSet name: {record_set.name} | @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    {field.name} (@id: {field.id}, type: {field.dataType})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set, field, and column references use their `@id` values.

In [ ]:
# Get all record set @id values
record_sets = [rs.id for rs in metadata.recordSets]

# Load each record set as a DataFrame
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns for the first record set
selected_record_set_id = record_sets[0]
print(f"Columns for RecordSet {selected_record_set_id}:\n", dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering, normalizing numeric fields, grouping, etc. All references use the @id values.

In [ ]:
# For demonstration, select a numeric field (@id) and a group field (@id)
rs = metadata.recordSets[0]
field_ids_types = [(f.id, f.dataType) for f in rs.fields]
print("Numeric fields with their @ids:", [ft for ft in field_ids_types if ft[1] in ('Float', 'Integer', 'Number')])
print("Categorical fields with their @ids:", [ft for ft in field_ids_types if ft[1] in ('Text',)])

# Set as example the first available numeric field and a categorical field
numeric_field_id = None
for field in rs.fields:
    if field.dataType in ('Float', 'Integer', 'Number'):
        numeric_field_id = field.id
        break
group_field_id = None
for field in rs.fields:
    if field.dataType == 'Text' and field.id != numeric_field_id:
        group_field_id = field.id
        break

print(f"Selected numeric_field_id: {numeric_field_id}")
print(f"Selected group_field_id: {group_field_id}")

# Simple EDA: filter, normalize, group
df = dataframes[selected_record_set_id]
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in {selected_record_set_id} with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped (mean) {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only show plot if a numeric field is available
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(9,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, showmeans=True)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- Successfully loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`.
- Record sets, field structure, and numeric/categorical variables have been identified by their `@id` as required by the Croissant schema.
- Demonstrated filtering, normalization, grouping, and visualizations for an initial exploratory analysis.
- For more advanced tasks, consult the field documentation, full Croissant schema, or extend with custom analytics.